In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/llm-classification-finetuning/sample_submission.csv
/kaggle/input/competitions/llm-classification-finetuning/train.csv
/kaggle/input/competitions/llm-classification-finetuning/test.csv
/kaggle/input/models/keras/deberta_v3/keras/deberta_v3_base_en/3/config.json
/kaggle/input/models/keras/deberta_v3/keras/deberta_v3_base_en/3/tokenizer.json
/kaggle/input/models/keras/deberta_v3/keras/deberta_v3_base_en/3/metadata.json
/kaggle/input/models/keras/deberta_v3/keras/deberta_v3_base_en/3/model.weights.h5
/kaggle/input/models/keras/deberta_v3/keras/deberta_v3_base_en/3/assets/tokenizer/vocabulary.spm


In [2]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import pandas as pd
import numpy as np
import tensorflow as tf
import keras_nlp
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
import warnings
warnings.filterwarnings('ignore')

print("TensorFlow version:", tf.__version__)
print("KerasNLP version:", keras_nlp.__version__)

# ============================================
# 1. LOAD DATA
# ============================================
print("\n" + "="*50)
print("Loading data...")
print("="*50)

train = pd.read_csv('/kaggle/input/competitions/llm-classification-finetuning/train.csv')
test = pd.read_csv('/kaggle/input/competitions/llm-classification-finetuning/test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

# ============================================
# 2. CLEAN AND PREPARE TEXT
# ============================================
print("\n" + "="*50)
print("Cleaning text data...")
print("="*50)

def clean_text(text):
    """Remove list brackets and quotes from text"""
    text = str(text)
    # Remove brackets if present
    if text.startswith('[') and text.endswith(']'):
        text = text[1:-1]
    # Remove quotes
    text = text.strip('"\'')
    # Replace escaped quotes
    text = text.replace('\\"', '"')
    return text

# Clean all text columns
for col in ['prompt', 'response_a', 'response_b']:
    train[col] = train[col].apply(clean_text)
    test[col] = test[col].apply(clean_text)

# Create formatted text input
def format_text(row):
    return f"""Prompt: {row['prompt']}

Response A: {row['response_a']}

Response B: {row['response_b']}"""

train['text'] = train.apply(format_text, axis=1)
test['text'] = test.apply(format_text, axis=1)

print("Sample text:")
print(train['text'].iloc[0][:300] + "...")

# ============================================
# 3. CREATE TARGETS
# ============================================
print("\n" + "="*50)
print("Creating targets...")
print("="*50)

train['label'] = train.apply(
    lambda x: 0 if x['winner_model_a'] == 1 
    else (1 if x['winner_model_b'] == 1 else 2), 
    axis=1
)

print(f"Class distribution:")
print(train['label'].value_counts(normalize=True).sort_index())
print(f"\nCounts:")
print(train['label'].value_counts().sort_index())

# ============================================
# 4. SPLIT DATA
# ============================================
print("\n" + "="*50)
print("Splitting data...")
print("="*50)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train['text'].values,
    train['label'].values,
    test_size=0.1,  # Smaller validation set for faster training
    random_state=42,
    stratify=train['label'].values
)

print(f"Train samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")

# ============================================
# 5. SIMPLE MODEL USING BUILT-IN CLASSIFIER
# ============================================
print("\n" + "="*50)
print("Building model...")
print("="*50)

# Use the built-in classifier which handles everything
# This is the simplest approach that should work without errors
model = keras_nlp.models.DebertaV3Classifier.from_preset(
    "deberta_v3_base_en",
    num_classes=3,
)

# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=2e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model summary:")
model.summary()

# ============================================
# 6. TRAIN THE MODEL
# ============================================
print("\n" + "="*50)
print("Training model...")
print("="*50)

# Simple training without complex callbacks
history = model.fit(
    train_texts,
    train_labels,
    validation_data=(val_texts, val_labels),
    batch_size=4,  # Small batch size for GPU memory
    epochs=3,
    verbose=1
)

# ============================================
# 7. EVALUATE ON VALIDATION
# ============================================
print("\n" + "="*50)
print("Evaluating on validation...")
print("="*50)

# Get predictions
val_preds = model.predict(val_texts)

# Calculate log loss
val_loss = log_loss(val_labels, val_preds)
print(f"Validation Log Loss: {val_loss:.4f}")

# Calculate accuracy
val_acc = (val_preds.argmax(axis=1) == val_labels).mean()
print(f"Validation Accuracy: {val_acc:.4f}")

# ============================================
# 8. CREATE SUBMISSION
# ============================================
print("\n" + "="*50)
print("Creating submission...")
print("="*50)

# Predict on test
test_preds = model.predict(test['text'].values)

# Create submission
submission = pd.DataFrame({
    'id': test['id'],
    'winner_model_a': test_preds[:, 0],
    'winner_model_b': test_preds[:, 1],
    'winner_tie': test_preds[:, 2]
})

# Save submission
submission.to_csv('submission.csv', index=False)
print("Submission saved!")
print("\nSubmission preview:")
print(submission.head())

# ============================================
# 9. IMPROVEMENT OPTIONS
# ============================================
print("\n" + "="*50)
print("Improvements you can try:")
print("="*50)
print("""
1. **Model Size**:
   - Use "deberta_v3_small_en" for faster training
   - Use "deberta_v3_large_en" for better accuracy (requires more memory)

2. **Training**:
   - Increase epochs to 5-10
   - Try different learning rates (1e-5, 3e-5, 5e-5)
   - Use learning rate scheduling

3. **Data**:
   - Add feature engineering (length differences, etc.)
   - Try different prompt formatting
   - Use data augmentation

4. **Cross-validation**:
   - Implement k-fold cross-validation
   - Ensemble multiple models

5. **Advanced**:
   - Use LoRA for efficient fine-tuning
   - Try different model architectures
   - Add model name embeddings
""")

print("\n" + "="*50)
print("Done! Ready to submit to competition.")
print("="*50)

2026-03-26 12:34:31.601727: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774528471.781505      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774528471.832323      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774528472.278732      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774528472.278772      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774528472.278775      24 computation_placer.cc:177] computation placer alr

TensorFlow version: 2.19.0
KerasNLP version: 0.21.1

Loading data...
Train shape: (57477, 9)
Test shape: (3, 4)

Cleaning text data...
Sample text:
Prompt: Is it morally right to try to have a certain percentage of females on managerial positions?","OK, does pineapple belong on a pizza? Relax and give me fun answer.

Response A: The question of whether it is morally right to aim for a certain percentage of females in managerial positions is a c...

Creating targets...
Class distribution:
label
0    0.349079
1    0.341911
2    0.309011
Name: proportion, dtype: float64

Counts:
label
0    20064
1    19652
2    17761
Name: count, dtype: int64

Splitting data...
Train samples: 51729
Validation samples: 5748

Building model...


I0000 00:00:1774528502.513454      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model summary:


Preprocessor: "deberta_v3_text_classifier_preprocessor"

...

 Trainable params: 184,424,451 (703.52 MB)

 Non-trainable params: 0 (0.00 B)


Training model...
Epoch 1/3


I0000 00:00:1774528556.178863      68 service.cc:152] XLA service 0x788a2802b170 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774528556.178900      68 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1774528563.674338      68 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1774528614.564047      68 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


12933/12933 ━━━━━━━━━━━━━━━━━━━━ 4062s 306ms/step - accuracy: 0.3165 - loss: 1.1093 - val_accuracy: 0.3090 - val_loss: 1.0986
Epoch 2/3
12933/12933 ━━━━━━━━━━━━━━━━━━━━ 3921s 303ms/step - accuracy: 0.3124 - loss: 1.0986 - val_accuracy: 0.3090 - val_loss: 1.0986
Epoch 3/3
12933/12933 ━━━━━━━━━━━━━━━━━━━━ 3918s 303ms/step - accuracy: 0.3116 - loss: 1.0986 - val_accuracy: 0.3090 - val_loss: 1.0986

Evaluating on validation...
180/180 ━━━━━━━━━━━━━━━━━━━━ 138s 729ms/step
Validation Log Loss: 15.9424
Validation Accuracy: 0.3090

Creating submission...
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
Submission saved!

Submission preview:
        id  winner_model_a  winner_model_b  winner_tie
0   136060       -0.765610       -0.911726   -0.500085
1   211333       -0.774694       -0.899050   -0.492069
2  1233961       -0.773321       -0.895836   -0.489915

Improvements you can try:

1. **Model Size**:
   - Use "deberta_v3_small_en" for faster training
   - Use "deberta_v3_large_en" for better accuracy (re